# LSTM Data Preprocessing Pipeline: Las Vegas and Phoenix

This notebook prepares the **Las Vegas** and **Phoenix** 2016–2023 hourly energy + weather datasets for the cross city LSTM experiment.

### Important decisions based on the previous pipeline

1. **February 29 is removed from both energy and weather data.**  
   The final timeline therefore goes directly from February 28 23:00 to March 1 00:00 in leap years.

2. The weather files already exclude February 29. The energy files still contain it, so it is removed before merging.

3. The weather files contain three radiation columns that are **100% NaN** in both cities. They are dropped rather than imputed.

4. `present_weather` contains the sentinel value `999999999` throughout the files and is therefore not treated as a normal numeric weather variable.

5. `snow_depth_cm` is constant at zero in both cities, so it carries no predictive information and is dropped.

6. `observation_indicator` is almost entirely zero and is treated as a source quality flag rather than an LSTM feature.

7. The old fixed energy outlier rule such as `energy > 400` is **not reused**. Las Vegas and Phoenix have very different load scales, so outliers are diagnosed using robust distribution checks instead.

8. No target-derived features such as lagged energy, rolling energy, or historical target means are used. This keeps the model focused on **weather + calendar/context → expected energy**.

9. Scalers are fitted **only on the source city's training period**. For cross city inference, the target city's data is transformed using the source city's scaler.

10. Sequences use a **24 hour window** and are created without crossing train/validation/test boundaries.

11. The notebook is deliberately reusable: the same feature definitions, column mapping, cleaning rules, and scaling logic are applied to both cities.


In [ ]:
# 1. Imports and configuration

from pathlib import Path
import json
import pickle
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
SEQUENCE_LENGTH = 24

BASE_DIR = Path.cwd()

DATA_FILES = {
    "Las Vegas": {
        "weather": BASE_DIR / "data" / "las_vegas_hourly_weather_2016_2023.csv",
        "energy": BASE_DIR / "data" / "las_vegas_hourly_energy.csv",
    },
    "Phoenix": {
        "weather": BASE_DIR / "data" / "phoenix_hourly_weather_2016_2023.csv",
        "energy": BASE_DIR / "data" / "phoenix_hourly_energy.csv",
    },
}

OUTPUT_DIR = BASE_DIR / "processed_lstm"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for city, files in DATA_FILES.items():
    for kind, path in files.items():
        print(city, kind, "->", path)


## 2. Raw data audit

The raw files were inspected before designing the cleaning rules.

### Observed structure

| City | Energy rows | Weather rows |
|---|---:|---:|
| Las Vegas | 70,128 | 70,080 |
| Phoenix | 70,128 | 70,080 |

The weather files contain exactly 8 × 8,760 = **70,080** rows because February 29 has already been removed.

The energy files contain 48 additional rows because the eight-year period includes leap days. Those 48 rows are removed during preprocessing.

Both energy files run from `2016-01-01 00:00:00` through `2023-12-31 23:00:00`.

The weather `hour` field is **1-based**: `hour=1` corresponds to timestamp `00:00`, `hour=24` corresponds to `23:00`. The notebook therefore treats the timestamp as the authoritative time field and only uses `hour` for validation.


In [ ]:
# 2.1 Load the four raw files

raw = {}

for city, files in DATA_FILES.items():
    raw[city] = {
        "weather": pd.read_csv(files["weather"]),
        "energy": pd.read_csv(files["energy"]),
    }

for city in raw:
    print(f"\n{'='*70}\n{city}")
    for kind, df in raw[city].items():
        print(f"{kind}: shape={df.shape}")
        print("columns:", list(df.columns))


In [ ]:
# 2.2 Detailed raw audit

def audit_raw(city, weather, energy):
    print(f"\n{'#'*80}\nRAW AUDIT: {city}\n{'#'*80}")

    for name, df in [("ENERGY", energy), ("WEATHER", weather)]:
        ts = pd.to_datetime(df["timestamp"], errors="coerce")
        print(f"\n{name}")
        print("Rows:", len(df))
        print("Timestamp range:", ts.min(), "->", ts.max())
        print("Invalid timestamps:", ts.isna().sum())
        print("Duplicate timestamps:", ts.duplicated().sum())

        sorted_ts = ts.sort_values()
        diffs = sorted_ts.diff().dropna()
        bad_gaps = (diffs != pd.Timedelta(hours=1)).sum()
        print("Non-1-hour transitions:", bad_gaps)

        feb29 = ((ts.dt.month == 2) & (ts.dt.day == 29)).sum()
        print("February 29 rows:", feb29)

        if "hourly_load_mw" in df:
            x = pd.to_numeric(df["hourly_load_mw"], errors="coerce")
            print("\nEnergy summary:")
            print(x.describe(percentiles=[0.001, 0.01, 0.50, 0.99, 0.999]).round(3))
            print("NaN:", x.isna().sum(), "negative:", (x < 0).sum(), "zero:", (x == 0).sum())
        else:
            missing = df.isna().sum()
            print("\nMissing values:")
            print(missing[missing > 0].sort_values(ascending=False).to_string())

            print("\nConstant / near-constant columns:")
            for c in df.columns:
                nunique = df[c].nunique(dropna=False)
                if nunique <= 3:
                    print(c, "nunique=", nunique, "top=", df[c].value_counts(dropna=False).head(3).to_dict())

            numeric = df.select_dtypes(include=np.number)
            print("\nNumeric ranges:")
            print(numeric.describe().T[["min", "max", "mean", "std"]].round(3).to_string())

for city in raw:
    audit_raw(city, raw[city]["weather"], raw[city]["energy"])


## 3. Cleaning strategy

### Timestamp and leap day

The timestamp is parsed and sorted. February 29 is removed from **both** datasets before alignment.

This is intentional. Otherwise the energy dataset would have 48 hours that have no corresponding weather observations.

### Weather quality fields

The following are handled based on what is actually present in the files:

* `global_horizontal_radiation_wh_m2`: 100% NaN → drop
* `direct_normal_radiation_wh_m2`: 100% NaN → drop
* `diffuse_horizontal_radiation_wh_m2`: 100% NaN → drop
* `snow_depth_cm`: constant zero → drop
* `present_weather`: source sentinel `999999999` → drop
* `observation_indicator`: source quality flag, not a physical predictor → drop

The remaining weather variables are retained initially and then assessed for variance and missingness.

### Outliers

We do **not** impose the old `400 kWh` threshold. The energy target is measured in MW and has very different scales between the two cities.

The notebook reports extreme quantiles and IQR based flags for inspection. It does not automatically delete legitimate extreme demand events merely because they are statistically unusual.


In [ ]:
# 3.1 Cleaning helpers

WEATHER_DROP_COLUMNS = [
    "date",  # timestamp is authoritative
    "hour",  # timestamp is authoritative; retained only for validation
    "global_horizontal_radiation_wh_m2",
    "direct_normal_radiation_wh_m2",
    "diffuse_horizontal_radiation_wh_m2",
    "snow_depth_cm",
    "present_weather",
    "observation_indicator",
]

def remove_feb29(df):
    ts = pd.to_datetime(df["timestamp"], errors="coerce")
    mask = ~((ts.dt.month == 2) & (ts.dt.day == 29))
    return df.loc[mask].copy()

def clean_energy(df):
    out = df.copy()
    out["timestamp"] = pd.to_datetime(out["timestamp"], errors="coerce")
    out = out.dropna(subset=["timestamp"])
    out = out.sort_values("timestamp")
    out = out.drop_duplicates(subset=["timestamp"], keep="first")
    out = remove_feb29(out)

    out["hourly_load_mw"] = pd.to_numeric(out["hourly_load_mw"], errors="coerce")
    out = out.dropna(subset=["hourly_load_mw"])

    # Physical sanity checks. Negative electricity demand is invalid here.
    out = out.loc[out["hourly_load_mw"] >= 0].copy()

    return out

def clean_weather(df):
    out = df.copy()
    out["timestamp"] = pd.to_datetime(out["timestamp"], errors="coerce")
    out = out.dropna(subset=["timestamp"])
    out = out.sort_values("timestamp")
    out = out.drop_duplicates(subset=["timestamp"], keep="first")
    out = remove_feb29(out)

    # Source sentinel used by present_weather.
    if "present_weather" in out:
        out.loc[out["present_weather"] == 999999999, "present_weather"] = np.nan

    # Validate source hour convention: 1 -> 00:00, ..., 24 -> 23:00.
    if "hour" in out:
        expected_hour = out["timestamp"].dt.hour + 1
        mismatch = (out["hour"] != expected_hour).sum()
        print("Hour/timestamp mismatches:", mismatch)

    # Remove unusable / non-predictive source columns.
    out = out.drop(columns=[c for c in WEATHER_DROP_COLUMNS if c in out.columns], errors="ignore")

    return out

cleaned = {}

for city in raw:
    energy = clean_energy(raw[city]["energy"])
    weather = clean_weather(raw[city]["weather"])

    cleaned[city] = {"energy": energy, "weather": weather}

    print(f"\n{city}")
    print("Clean energy:", energy.shape)
    print("Clean weather:", weather.shape)


In [ ]:
# 3.2 Verify the leap-day transition explicitly

for city in cleaned:
    print(f"\n{city}")
    weather_ts = cleaned[city]["weather"]["timestamp"]
    energy_ts = cleaned[city]["energy"]["timestamp"]

    for name, ts in [("energy", energy_ts), ("weather", weather_ts)]:
        ts = ts.sort_values()
        leap_years = [2016, 2020]
        for year in leap_years:
            before = ts[(ts.dt.year == year) & (ts.dt.month == 2) & (ts.dt.day == 28)].max()
            after = ts[(ts.dt.year == year) & (ts.dt.month == 3) & (ts.dt.day == 1)].min()
            print(name, year, "before:", before, "after:", after,
                  "gap:", after - before)


## 4. Merge energy and weather

Both datasets are aligned on the cleaned hourly `timestamp`.

An **inner merge** is used after the explicit leap-day removal. This makes it impossible for an energy observation without a corresponding weather observation to silently enter the LSTM dataset.

The notebook checks:

* duplicate timestamps after merging
* missing values
* hourly continuity
* expected row count
* whether energy and weather cover the same time range


In [ ]:
# 4.1 Merge and validate

merged = {}

for city in cleaned:
    energy = cleaned[city]["energy"]
    weather = cleaned[city]["weather"]

    df = pd.merge(
        energy,
        weather,
        on="timestamp",
        how="inner",
        validate="one_to_one",
    ).sort_values("timestamp").reset_index(drop=True)

    merged[city] = df

    ts = df["timestamp"]
    diffs = ts.diff().dropna()

    print(f"\n{'='*70}\n{city}")
    print("Merged shape:", df.shape)
    print("Timestamp range:", ts.min(), "->", ts.max())
    print("Duplicate timestamps:", ts.duplicated().sum())
    print("Non-1-hour transitions:", (diffs != pd.Timedelta(hours=1)).sum())
    print("February 29:", ((ts.dt.month == 2) & (ts.dt.day == 29)).sum())
    print("Total missing cells:", int(df.isna().sum().sum()))


## 5. Missing value treatment

At this point the data is merged and we can see the actual missingness relevant to the model.

The notebook does **not** use blind global filling.

For continuous meteorological variables, short internal gaps can be linearly interpolated after the timeline has been regularized. The target is never fabricated by interpolation.

If the weather data contains no missing values, no interpolation is performed.

The final missing value report is retained as an audit artifact.


In [ ]:
# 5.1 Regularize the hourly timeline and fill only weather gaps

def regularize_and_impute_weather(df):
    out = df.copy().sort_values("timestamp").set_index("timestamp")

    # Reindex to the observed start/end at an hourly frequency.
    full_index = pd.date_range(out.index.min(), out.index.max(), freq="h")
    out = out.reindex(full_index)
    out.index.name = "timestamp"

    target_col = "hourly_load_mw"
    weather_cols = [c for c in out.columns if c != target_col]

    # Never interpolate the target.
    before_target_missing = out[target_col].isna().sum()

    # Time interpolation for continuous weather measurements.
    numeric_weather = out[weather_cols].select_dtypes(include=np.number).columns
    out[numeric_weather] = out[numeric_weather].interpolate(
        method="time",
        limit_direction="both",
    )

    after_target_missing = out[target_col].isna().sum()

    return out.reset_index(), before_target_missing, after_target_missing

for city in merged:
    merged[city], before_target_missing, after_target_missing = regularize_and_impute_weather(merged[city])
    print(city, "target missing before/after weather imputation:", before_target_missing, after_target_missing)
    print("Remaining missing cells:", int(merged[city].isna().sum().sum()))


## 6. Energy distribution and outlier diagnostics

The raw audit showed:

* Las Vegas load is roughly 2.1–6.8 GW in the supplied data.
* Phoenix load is roughly 3.3–13.2 GW in the supplied data.
* Neither dataset contains negative or zero target values.
* The two cities have substantially different load scales.

Therefore a universal numeric outlier cutoff would be inappropriate.

The following diagnostics are used instead:

* 0.1%, 1%, 50%, 99%, 99.9% quantiles
* IQR based statistical flags
* yearly and monthly distributions
* optional visualization

Statistical flags are **diagnostic only**. They are not automatically deleted.


In [ ]:
# 6.1 Target diagnostics

def energy_outlier_report(df):
    x = df["hourly_load_mw"]
    q1, q3 = x.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    return {
        "min": x.min(),
        "max": x.max(),
        "q01": x.quantile(0.01),
        "q50": x.quantile(0.50),
        "q99": x.quantile(0.99),
        "q999": x.quantile(0.999),
        "iqr_lower": lower,
        "iqr_upper": upper,
        "iqr_flag_count": int(((x < lower) | (x > upper)).sum()),
        "iqr_flag_pct": float(((x < lower) | (x > upper)).mean() * 100),
    }

for city, df in merged.items():
    print(city)
    print(pd.Series(energy_outlier_report(df)).round(3))


In [ ]:
# 6.2 Load distributions

for city, df in merged.items():
    plt.figure(figsize=(10, 5))
    plt.hist(df["hourly_load_mw"], bins=80)
    plt.title(f"{city} hourly electricity load")
    plt.xlabel("Hourly load (MW)")
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.show()


## 7. Feature engineering

The feature design is kept identical across Las Vegas and Phoenix.

### Calendar features

* hour
* day of week
* month
* weekend
* day of year
* season

Cyclical encodings are used for periodic variables:

* `sin_hour`, `cos_hour`
* `sin_dow`, `cos_dow`
* `sin_month`, `cos_month`
* `sin_doy`, `cos_doy`

### Weather features

The core physical variables are retained:

* temperature
* dew point
* relative humidity
* station pressure
* wind direction
* wind speed
* horizontal visibility
* ceiling height
* precipitable water
* aerosol optical depth
* hourly precipitation

Additional nonlinear/context features:

* temperature squared
* temperature × relative humidity
* cooling degree hour relative to 18.3°C
* heating degree hour relative to 18.3°C

No energy-derived feature is introduced.


In [ ]:
# 7.1 Feature engineering

def add_features(df):
    out = df.copy().sort_values("timestamp").reset_index(drop=True)
    ts = out["timestamp"]

    # Calendar
    out["hour"] = ts.dt.hour
    out["day_of_week"] = ts.dt.dayofweek
    out["month"] = ts.dt.month
    out["day_of_year"] = ts.dt.dayofyear
    out["is_weekend"] = (ts.dt.dayofweek >= 5).astype(np.int8)

    # Season: meteorological seasons
    out["season"] = ts.dt.month.map({
        12: "winter", 1: "winter", 2: "winter",
        3: "spring", 4: "spring", 5: "spring",
        6: "summer", 7: "summer", 8: "summer",
        9: "autumn", 10: "autumn", 11: "autumn",
    })

    season_code = {"winter": 0, "spring": 1, "summer": 2, "autumn": 3}
    out["season_code"] = out["season"].map(season_code).astype(np.int8)

    # Cyclical encodings
    out["sin_hour"] = np.sin(2 * np.pi * out["hour"] / 24)
    out["cos_hour"] = np.cos(2 * np.pi * out["hour"] / 24)

    out["sin_dow"] = np.sin(2 * np.pi * out["day_of_week"] / 7)
    out["cos_dow"] = np.cos(2 * np.pi * out["day_of_week"] / 7)

    out["sin_month"] = np.sin(2 * np.pi * (out["month"] - 1) / 12)
    out["cos_month"] = np.cos(2 * np.pi * (out["month"] - 1) / 12)

    out["sin_doy"] = np.sin(2 * np.pi * (out["day_of_year"] - 1) / 365)
    out["cos_doy"] = np.cos(2 * np.pi * (out["day_of_year"] - 1) / 365)

    # Nonlinear weather/context terms
    out["temperature_squared"] = out["temperature_c"] ** 2
    out["temperature_humidity_interaction"] = (
        out["temperature_c"] * out["relative_humidity_percent"]
    )

    BASE_TEMP_C = 18.3
    out["cooling_degree_hour"] = np.maximum(out["temperature_c"] - BASE_TEMP_C, 0)
    out["heating_degree_hour"] = np.maximum(BASE_TEMP_C - out["temperature_c"], 0)

    return out

featured = {city: add_features(df) for city, df in merged.items()}

for city, df in featured.items():
    print(city, df.shape)
    print(df.head(2).T)


## 8. Feature consistency check

For cross city transfer, the source and target cities must have the **same semantic feature set**.

This cell checks that the numeric model feature columns exist in both datasets.

`season` is retained as metadata, while `season_code` is the numeric representation used by the model.


In [ ]:
# 8.1 Define the final model feature set

TARGET = "hourly_load_mw"

MODEL_FEATURES = [
    # Raw weather
    "temperature_c",
    "dew_point_c",
    "relative_humidity_percent",
    "station_pressure_kpa",
    "wind_direction_deg",
    "wind_speed_ms",
    "horizontal_visibility_km",
    "ceiling_height_m",
    "precipitable_water_mm",
    "broadband_aerosol_optical_depth",
    "hourly_precipitation_cm",

    # Calendar/context
    "hour",
    "day_of_week",
    "month",
    "is_weekend",
    "season_code",
    "sin_hour",
    "cos_hour",
    "sin_dow",
    "cos_dow",
    "sin_month",
    "cos_month",
    "sin_doy",
    "cos_doy",

    # Nonlinear weather terms
    "temperature_squared",
    "temperature_humidity_interaction",
    "cooling_degree_hour",
    "heating_degree_hour",
]

for city, df in featured.items():
    missing_features = [c for c in MODEL_FEATURES if c not in df.columns]
    print(city, "missing model features:", missing_features)

assert set(MODEL_FEATURES).issubset(featured["Las Vegas"].columns)
assert set(MODEL_FEATURES).issubset(featured["Phoenix"].columns)
assert len(MODEL_FEATURES) == len(set(MODEL_FEATURES))

print("\nNumber of model features:", len(MODEL_FEATURES))
print(MODEL_FEATURES)


## 9. Time based train / validation / test split

Because the data spans eight years, a chronological split is preferable to a random row split.

Default:

* first 70% of the timeline → training
* next 15% → validation
* final 15% → test

This prevents future observations from being used to train the model.

For the cross city experiment, the **source city's training data determines the scaler**. The target city is never used to fit that scaler.


In [ ]:
# 9.1 Chronological split

def chronological_split(df, train_frac=0.70, val_frac=0.15):
    n = len(df)
    train_end = int(n * train_frac)
    val_end = train_end + int(n * val_frac)

    train = df.iloc[:train_end].copy()
    val = df.iloc[train_end:val_end].copy()
    test = df.iloc[val_end:].copy()

    return train, val, test

splits = {}

for city, df in featured.items():
    train, val, test = chronological_split(df)

    splits[city] = {
        "train": train,
        "val": val,
        "test": test,
    }

    print(f"\n{city}")
    for name, part in splits[city].items():
        print(
            name,
            len(part),
            part["timestamp"].min(),
            "->",
            part["timestamp"].max()
        )


## 10. Leakage safe scaling

A separate scaler is fitted on each source city's **training feature matrix only**.

For example:

```text
Las Vegas train
      ↓
fit LV scaler
      ↓
LV validation / LV test / Phoenix cross city
      ↓
transform only
```

The same principle works in the opposite direction if Phoenix is used as the source city.

The target `hourly_load_mw` is also scaled separately for model training. The target scaler is fitted only on the source training target and is used to inverse transform predictions.


In [ ]:
# 10.1 Fit source-city scalers and transform local splits

def fit_source_scalers(train_df):
    x_scaler = StandardScaler()
    y_scaler = StandardScaler()

    x_scaler.fit(train_df[MODEL_FEATURES].astype(np.float32))
    y_scaler.fit(train_df[[TARGET]].astype(np.float32))

    return x_scaler, y_scaler

scaled = {}
scalers = {}

for city in splits:
    x_scaler, y_scaler = fit_source_scalers(splits[city]["train"])
    scalers[city] = {"X": x_scaler, "y": y_scaler}

    scaled[city] = {}

    for split_name, df in splits[city].items():
        out = df[["timestamp", TARGET] + MODEL_FEATURES].copy()
        out[MODEL_FEATURES] = x_scaler.transform(
            out[MODEL_FEATURES].astype(np.float32)
        ).astype(np.float32)
        out[TARGET] = y_scaler.transform(
            out[[TARGET]].astype(np.float32)
        ).astype(np.float32).ravel()

        scaled[city][split_name] = out

    print(city, "scalers fitted only on", len(splits[city]["train"]), "training rows")


## 11. Cross city preprocessing

This is the important transfer setup.

### Experiment A

```text
Las Vegas train → fit scalers → train LSTM
Phoenix → transform using Las Vegas scalers → cross city test
```

### Experiment B

```text
Phoenix train → fit scalers → train LSTM
Las Vegas → transform using Phoenix scalers → cross city test
```

The target city is **not** used to fit the source scaler.

Because the model uses only weather and calendar/context features, the target city's previous energy values are never needed to create an input sequence.


In [ ]:
# 11.1 Create cross-city transformed data

cross_city = {}

for source_city, target_city in [
    ("Las Vegas", "Phoenix"),
    ("Phoenix", "Las Vegas"),
]:
    x_scaler = scalers[source_city]["X"]
    y_scaler = scalers[source_city]["y"]

    transformed_target = featured[target_city][["timestamp", TARGET] + MODEL_FEATURES].copy()

    transformed_target[MODEL_FEATURES] = x_scaler.transform(
        transformed_target[MODEL_FEATURES].astype(np.float32)
    ).astype(np.float32)

    transformed_target[TARGET] = y_scaler.transform(
        transformed_target[[TARGET]].astype(np.float32)
    ).astype(np.float32).ravel()

    cross_city[f"{source_city}_to_{target_city}"] = transformed_target

    print(
        f"{source_city} -> {target_city}:",
        transformed_target.shape,
        "| source scaler:", source_city
    )


## 12. Memory efficient 24 hour sequence generation

The previous project encountered RAM pressure when constructing large sequence arrays.

This notebook therefore provides two options:

1. A NumPy sequence builder for smaller debugging runs.
2. A `tf.data` pipeline that constructs windows lazily for the actual GPU training workflow.

The target is the energy value at the **last timestamp of each 24 hour window**.

No window is allowed to cross a train/validation/test boundary.


In [ ]:
# 12.1 NumPy sequence builder for debugging / validation

def make_sequences_numpy(df, sequence_length=24):
    X = df[MODEL_FEATURES].to_numpy(dtype=np.float32)
    y = df[TARGET].to_numpy(dtype=np.float32)

    n = len(df) - sequence_length + 1
    if n <= 0:
        raise ValueError("Not enough rows for the requested sequence length.")

    X_seq = np.empty((n, sequence_length, len(MODEL_FEATURES)), dtype=np.float32)
    y_seq = np.empty(n, dtype=np.float32)

    for i in range(n):
        j = i + sequence_length
        X_seq[i] = X[i:j]
        y_seq[i] = y[j - 1]

    return X_seq, y_seq

# Small sanity check
for city in scaled:
    X_dbg, y_dbg = make_sequences_numpy(
        scaled[city]["train"].head(200),
        SEQUENCE_LENGTH
    )
    print(city, "debug sequence shape:", X_dbg.shape, y_dbg.shape)


In [ ]:
# 12.2 TensorFlow dataset builder for the actual LSTM workflow

# TensorFlow is imported here so the preprocessing notebook can still be
# used without requiring TensorFlow for the earlier audit cells.

import tensorflow as tf

print("TensorFlow version:", tf.__version__)
print("Visible GPUs:", tf.config.list_physical_devices("GPU"))

def make_tf_dataset(df, batch_size=64, sequence_length=24, shuffle=False):
    X = df[MODEL_FEATURES].to_numpy(dtype=np.float32)
    y = df[TARGET].to_numpy(dtype=np.float32)

    # Window the feature matrix and target together.
    ds = tf.data.Dataset.from_tensor_slices((X, y))

    ds = ds.window(
        size=sequence_length,
        shift=1,
        stride=1,
        drop_remainder=True
    )

    ds = ds.flat_map(
        lambda x, y: tf.data.Dataset.zip((
            x.batch(sequence_length, drop_remainder=True),
            y.batch(sequence_length, drop_remainder=True)
        ))
    )

    ds = ds.map(
        lambda x, y: (x, y[-1]),
        num_parallel_calls=tf.data.AUTOTUNE
    )

    if shuffle:
        ds = ds.shuffle(
            buffer_size=min(len(df), 10000),
            seed=RANDOM_STATE,
            reshuffle_each_iteration=True
        )

    ds = ds.batch(batch_size)
    ds = ds.prefetch(tf.data.AUTOTUNE)

    return ds

# Example only: construct datasets after GPU environment validation.
for city in scaled:
    train_ds = make_tf_dataset(scaled[city]["train"], shuffle=True)
    val_ds = make_tf_dataset(scaled[city]["val"], shuffle=False)

    print(
        city,
        "train batches:",
        tf.data.experimental.cardinality(train_ds).numpy(),
        "validation batches:",
        tf.data.experimental.cardinality(val_ds).numpy()
    )


## 13. Saving preprocessing outputs

The notebook saves:

* cleaned + feature engineered city datasets
* local train/validation/test CSVs
* source city feature scaler
* source city target scaler
* feature list and preprocessing metadata
* cross city transformed datasets

The actual large LSTM sequence arrays are **not automatically saved as giant NumPy files**. They can be generated through `tf.data` when training, which is safer for the RTX 3050 + 16 GB RAM setup.

The scaler files are source specific so that cross city experiments remain reproducible.


In [ ]:
# 13.1 Save processed data, scalers and metadata

for city, df in featured.items():
    city_key = city.lower().replace(" ", "_")
    city_dir = OUTPUT_DIR / city_key
    city_dir.mkdir(parents=True, exist_ok=True)

    df.to_csv(city_dir / f"{city_key}_featured.csv", index=False)

    for split_name, split_df in splits[city].items():
        split_df.to_csv(
            city_dir / f"{city_key}_{split_name}.csv",
            index=False
        )

    with open(city_dir / "feature_scaler.pkl", "wb") as f:
        pickle.dump(scalers[city]["X"], f)

    with open(city_dir / "target_scaler.pkl", "wb") as f:
        pickle.dump(scalers[city]["y"], f)

    metadata = {
        "city": city,
        "target": TARGET,
        "model_features": MODEL_FEATURES,
        "sequence_length": SEQUENCE_LENGTH,
        "train_fraction": 0.70,
        "validation_fraction": 0.15,
        "test_fraction": 0.15,
        "leap_day_removed": True,
        "energy_outlier_fixed_threshold_used": False,
        "target_derived_features_used": False,
        "scaler_fit_on": "source city training split only",
    }

    with open(city_dir / "preprocessing_metadata.json", "w") as f:
        json.dump(metadata, f, indent=2)

    print("Saved:", city_dir)

for key, df in cross_city.items():
    source, target = key.split("_to_")
    cross_dir = OUTPUT_DIR / "cross_city"
    cross_dir.mkdir(parents=True, exist_ok=True)

    df.to_csv(
        cross_dir / f"{source.lower().replace(' ', '_')}_to_{target.lower().replace(' ', '_')}.csv",
        index=False
    )

print("\nAll preprocessing outputs saved under:", OUTPUT_DIR.resolve())


## 14. Final validation checklist

Before starting LSTM training, run the following checks.

### Dataset integrity

* [ ] No February 29 rows
* [ ] No duplicate timestamps
* [ ] No invalid timestamps
* [ ] No unintended missing target values
* [ ] Energy and weather aligned one-to-one
* [ ] Timestamp is hourly within each continuous segment
* [ ] No accidental 400 MW / 400 kWh style universal outlier rule

### Feature integrity

* [ ] Same feature list for Las Vegas and Phoenix
* [ ] Radiation columns excluded because they are entirely missing
* [ ] Constant snow depth excluded
* [ ] Source quality fields excluded
* [ ] No target-derived features
* [ ] Calendar features generated identically

### Leakage prevention

* [ ] Chronological train/validation/test split
* [ ] Feature scaler fitted only on source training data
* [ ] Target scaler fitted only on source training data
* [ ] Cross city data transformed with the source scaler
* [ ] Sequence windows do not cross split boundaries

### LSTM readiness

* [ ] Sequence length = 24
* [ ] Float32 inputs
* [ ] `tf.data` used for memory efficiency
* [ ] GPU visible in TensorFlow
* [ ] One model trained at a time on the RTX 3050

After these checks pass, the resulting datasets are ready for the separate LSTM training notebook.


In [ ]:
# 14.1 Automated final checks

for city, df in featured.items():
    ts = df["timestamp"].sort_values()

    assert ((ts.dt.month == 2) & (ts.dt.day == 29)).sum() == 0
    assert ts.duplicated().sum() == 0
    assert df[TARGET].notna().all()
    assert (df[TARGET] >= 0).all()
    assert all(c in df.columns for c in MODEL_FEATURES)

print("All automated final checks passed.")
print("\nCities prepared:", list(featured.keys()))
print("Feature count:", len(MODEL_FEATURES))
print("Sequence length:", SEQUENCE_LENGTH)
